# Notebook 05 — Revisi & Retain
## Sistem CBR Putusan Kasasi Peradilan Anak ABH · Mahkamah Agung RI

Notebook ini menjalankan **Tahap Revise & Retain CBR**:
memverifikasi prediksi dan menyimpan kasus terverifikasi ke case base.

### Dua alur retain

| Alur | Sumber | Tujuan |
|------|--------|--------|
| **Alur A** (Cell 3–6) | `predictions.csv` dari test set | Benchmark — update case base dengan solusi terverifikasi |
| **Alur B** (Cell 7) | 5 kasus baru konstruktif dari NB04 | Simulasi open-world — retain kasus yang benar-benar baru |

**Alur A** menggunakan kasus test set (closed-world) — kasus sudah ada di case base,
retain memperkaya solusinya. **Alur B** mensimulasikan retain kasus yang betul-betul
baru (open-world) — hanya kasus yang prediksinya benar dan confidence tinggi yang masuk.


## 1 · Import & Load Data

In [8]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ── Resolusi ROOT_DIR ───────────────────────────────────────────────
# Agar path data/ tetap valid baik notebook dijalankan dari root project
# maupun dari dalam folder notebook/ (working directory kernel kadang
# ikut lokasi file .ipynb). Cari ke atas folder yang punya subfolder data/;
# kalau belum ada (run pertama kali), fallback: naik 1 level jika cwd = "notebook".
def _tentukan_root_dir() -> Path:
    cwd = Path.cwd().resolve()
    p = cwd
    for _ in range(4):
        if (p / "data").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return cwd.parent if cwd.name == "notebook" else cwd

ROOT_DIR = _tentukan_root_dir()

DATA_DIR      = ROOT_DIR / 'data'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
RESULTS_DIR   = ROOT_DIR / 'data' / 'results'
EVAL_DIR      = ROOT_DIR / 'data' / 'eval'
METRICS_DIR   = ROOT_DIR / 'data' / 'eval' / 'metrics'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ── Load case base saat ini ───────────────────────────────────────────
with open(PROCESSED_DIR / 'case_base.json', encoding='utf-8') as f:
    case_base = json.load(f)
df_cb = pd.DataFrame(case_base)
existing_ids = set(df_cb['case_id'].tolist())

# Index case_base by case_id — sumber utama teks_query dan metadata
# queries.json hanya punya 10 kasus (sample eval), case_base punya semua 150
case_base_map = {e['case_id']: e for e in case_base}

# ── Load predictions dari NB04 ────────────────────────────────────────
df_pred = pd.read_csv(RESULTS_DIR / 'predictions.csv', encoding='utf-8-sig')

# ── Load queries.json (opsional, hanya untuk nomor_perkara fallback) ──
with open(METRICS_DIR / 'queries.json', encoding='utf-8') as f:
    queries_eval = json.load(f)
query_map_by_caseid = {q['case_id_sumber']: q for q in queries_eval}

print(f'Case base saat ini     : {len(case_base)} kasus (index utama)')
print(f'queries.json           : {len(queries_eval)} kasus (hanya sample eval)')
print(f'Total prediksi NB04    : {len(df_pred)} kasus')
print(f'Kolom predictions.csv  : {list(df_pred.columns)}')
print()
print('Distribusi outcome kasasi:')
print(df_pred['ground_truth'].value_counts().to_string())

Case base saat ini     : 150 kasus (index utama)
queries.json           : 10 kasus (hanya sample eval)
Total prediksi NB04    : 28 kasus
Kolom predictions.csv  : ['query_id', 'nomor_perkara', 'ground_truth', 'predicted_solution', 'predicted_outcome_vote', 'predicted_outcome_weight', 'predicted_outcome_svm', 'predicted_outcome_hybrid', 'svm_confidence', 'svm_proba', 'top_5_case_ids', 'top_5_scores', 'vote_breakdown', 'weight_breakdown']

Distribusi outcome kasasi:
ground_truth
DITOLAK       25
DIPERBAIKI     2
DIKABULKAN     1


## 2 · Filter Kasus Verified Correct

Kasus yang di-retain adalah kasus di mana:
- Prediksi Hybrid == Ground Truth (solusi terbukti tepat)
- Confidence SVM >= 0.40 (model cukup yakin)
- Belum ada di case base (tidak duplikat)

Kasus dari **test set** yang prediksinya benar diperlakukan sebagai
"kasus baru yang solusinya telah terverifikasi" — cocok untuk di-retain.

In [9]:
# ── Kriteria retain ───────────────────────────────────────────────────
# Konteks: predictions.csv berisi kasus dari TEST SET yang berasal dari
# case base sendiri. Siklus CBR yang sesungguhnya adalah:
#   kasus BARU (dari luar) → retrieve → reuse → solusi → verifikasi → retain
#
# Karena kasus uji berasal dari case base, retain di sini dimaknai sebagai:
#   "Perbarui entry case base yang ada dengan amar prediksi yang terverifikasi"
# Ini memperkaya case base dengan solusi yang telah dikonfirmasi benar,
# sehingga retrieval berikutnya punya referensi solusi yang lebih lengkap.

CONFIDENCE_THRESHOLD = 0.40
KOLOM_PRED = 'predicted_outcome_hybrid'
KOLOM_CONF = 'svm_confidence'
KOLOM_GT   = 'ground_truth'
KOLOM_QID  = 'query_id'

if KOLOM_PRED not in df_pred.columns:
    alt = [c for c in df_pred.columns if 'hybrid' in c.lower()]
    KOLOM_PRED = alt[0] if alt else 'predicted_outcome_weight'
    print(f'[WARN] Kolom hybrid tidak ditemukan, pakai: {KOLOM_PRED}')

# Filter 1: prediksi benar (solusi terverifikasi)
mask_benar = df_pred[KOLOM_PRED] == df_pred[KOLOM_GT]

# Filter 2: confidence SVM cukup tinggi
if KOLOM_CONF in df_pred.columns:
    mask_conf = df_pred[KOLOM_CONF] >= CONFIDENCE_THRESHOLD
else:
    mask_conf = pd.Series([True] * len(df_pred))
    print('[WARN] Kolom svm_confidence tidak ditemukan, skip filter confidence')

# TIDAK ada filter mask_baru:
# Retain = update/enrichment entry yang sudah ada dengan solusi terverifikasi
# Kasus yang SUDAH ada → perbarui field amar_putusan & _retain_* nya
# Kasus yang BELUM ada → tambahkan sebagai entry baru (untuk iterasi berikutnya)
mask_sudah_ada = df_pred[KOLOM_QID].isin(existing_ids)
mask_belum_ada = ~mask_sudah_ada

df_retain_update = df_pred[mask_benar & mask_conf & mask_sudah_ada].copy()  # update
df_retain_baru   = df_pred[mask_benar & mask_conf & mask_belum_ada].copy()  # tambah baru
df_retain_kandidat = pd.concat([df_retain_update, df_retain_baru], ignore_index=True)

print(f'Total prediksi             : {len(df_pred)}')
print(f'  Prediksi benar           : {mask_benar.sum()}')
print(f'  Confidence >= {CONFIDENCE_THRESHOLD}        : {mask_conf.sum()}')
print(f'  Sudah ada di case base   : {mask_sudah_ada.sum()}')
print(f'  Belum ada (entry baru)   : {mask_belum_ada.sum()}')
print()
print(f'  → Akan di-UPDATE (enrichment) : {len(df_retain_update)} kasus')
print(f'  → Akan DITAMBAH (entry baru)  : {len(df_retain_baru)} kasus')
print(f'  → Total kandidat retain       : {len(df_retain_kandidat)} kasus')
print()
print('Distribusi outcome kandidat:')
if len(df_retain_kandidat) > 0:
    print(df_retain_kandidat[KOLOM_GT].value_counts().to_string())

Total prediksi             : 28
  Prediksi benar           : 28
  Confidence >= 0.4        : 28
  Sudah ada di case base   : 28
  Belum ada (entry baru)   : 0

  → Akan di-UPDATE (enrichment) : 28 kasus
  → Akan DITAMBAH (entry baru)  : 0 kasus
  → Total kandidat retain       : 28 kasus

Distribusi outcome kandidat:
ground_truth
DITOLAK       25
DIPERBAIKI     2
DIKABULKAN     1


## 3 · Susun Entry Case Base Baru

Tiap kasus baru direpresentasikan dalam format yang identik dengan
entry di `case_base.json` sehingga NB03 & NB04 dapat langsung
membacanya tanpa perubahan.

In [10]:
def bangun_entry_baru(row, case_base_map, query_map_by_caseid, iterasi=2):
    """
    Susun entry case base baru dari baris predictions.csv.

    Sumber data (prioritas):
    1. case_base_map (case_base.json) — punya semua 150 kasus, sumber utama
    2. query_map_by_caseid (queries.json) — hanya 10 kasus, fallback minor
    """
    qid      = str(row.get(KOLOM_QID, ''))
    cb_entry = case_base_map.get(qid, {})
    teks_q   = cb_entry.get('teks_query', '')

    if not teks_q:
        q_entry = query_map_by_caseid.get(qid, {})
        teks_q  = q_entry.get('query_teks', '')

    if not teks_q:
        return None

    no_perk   = cb_entry.get('nomor_perkara', qid)
    top5_raw  = str(row.get('top_5_case_ids', ''))
    top5_list = [x.strip() for x in top5_raw.split(',') if x.strip()]

    return {
        'case_id'           : qid,
        'nomor_perkara'     : no_perk,
        'tahun'             : cb_entry.get('tahun', str(datetime.now().year)),
        'outcome_kasasi'    : row[KOLOM_GT],
        'teks_query'        : teks_q,
        'pasal_didakwakan'  : cb_entry.get('pasal_didakwakan', ''),
        'duduk_perkara'     : cb_entry.get('duduk_perkara', ''),
        'memori_kasasi'     : cb_entry.get('memori_kasasi', ''),
        'pertimbangan_hukum': cb_entry.get('pertimbangan_hukum', ''),
        'amar_putusan'      : str(row.get('predicted_solution',
                                cb_entry.get('amar_putusan', ''))),
        'vonis'             : cb_entry.get('vonis', ''),
        'word_count'        : len(teks_q.split()),
        'char_count'        : len(teks_q),
        '_retain_iterasi'   : iterasi,
        '_retain_timestamp' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        '_retain_sumber'    : 'predictions_hybrid_verified',
        '_retain_top5_ref'  : top5_list,
        '_retain_svm_conf'  : float(row.get(KOLOM_CONF, 0.0)),
    }

print('Fungsi bangun_entry_baru() siap.')
print('Sumber utama: case_base_map (150 kasus) + fallback queries.json (10 kasus)')

Fungsi bangun_entry_baru() siap.
Sumber utama: case_base_map (150 kasus) + fallback queries.json (10 kasus)


## 4 · Deduplikasi & Tambahkan ke Case Base

Cek sekali lagi apakah ada kasus yang sudah masuk case base
(misalnya dari iterasi sebelumnya), lalu merge dan simpan.

In [11]:
# ── Pisahkan: kasus untuk di-update vs kasus baru ────────────────────
id_update = set(df_retain_update[KOLOM_QID].tolist()) if len(df_retain_update) > 0 else set()
id_baru   = set(df_retain_baru[KOLOM_QID].tolist())   if len(df_retain_baru)   > 0 else set()

# ── UPDATE: perbarui entry yang sudah ada dengan amar terverifikasi ───
n_updated = 0
case_base_updated = []
for entry in case_base:
    cid = entry['case_id']
    if cid in id_update:
        row_pred = df_retain_update[df_retain_update[KOLOM_QID] == cid]
        if not row_pred.empty:
            r = row_pred.iloc[0]
            entry = entry.copy()
            entry['amar_putusan']       = str(r.get('predicted_solution',
                                            entry.get('amar_putusan', '')))
            entry['_retain_iterasi']    = 2
            entry['_retain_timestamp']  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            entry['_retain_sumber']     = 'predictions_hybrid_verified'
            entry['_retain_svm_conf']   = float(r.get(KOLOM_CONF, 0.0))
            entry['_retain_outcome_ok'] = True
            n_updated += 1
    case_base_updated.append(entry)

# ── TAMBAH: entry benar-benar baru (kasus dari luar case base) ────────
entri_baru = []
entri_skip = []
for _, row in df_retain_baru.iterrows():
    entry = bangun_entry_baru(row, case_base_map, query_map_by_caseid, iterasi=2)
    if entry:
        entri_baru.append(entry)
    else:
        entri_skip.append(str(row.get(KOLOM_QID, '?')))

case_base_baru = case_base_updated + entri_baru

# ── Backup & simpan ───────────────────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_path = PROCESSED_DIR / f'case_base_backup_{ts}.json'
with open(backup_path, 'w', encoding='utf-8') as f:
    json.dump(case_base, f, ensure_ascii=False, indent=2)
print(f'Backup case base lama  -> {backup_path}')

cb_path = PROCESSED_DIR / 'case_base.json'
with open(cb_path, 'w', encoding='utf-8') as f:
    json.dump(case_base_baru, f, ensure_ascii=False, indent=2)

print(f'case_base.json diperbarui -> {cb_path}')
print()
print(f'Sebelum retain         : {len(case_base)} kasus')
print(f'Di-update (enrichment) : {n_updated} kasus  ← amar_putusan terverifikasi')
print(f'Ditambah (entry baru)  : {len(entri_baru)} kasus  ← kasus dari luar case base')
print(f'Di-skip (no teks)      : {len(entri_skip)}')
print(f'Total sekarang         : {len(case_base_baru)} kasus')

if n_updated > 0:
    print()
    print('Contoh entry yang di-update:')
    contoh = [e for e in case_base_baru if e.get('_retain_iterasi') == 2][:3]
    for e in contoh:
        amar_preview = str(e.get('amar_putusan',''))[:60].replace('\n',' ')
        print(f"  {e['case_id']:<12} | {e['outcome_kasasi']:<14} | amar: {amar_preview}")

Backup case base lama  -> D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\processed\case_base_backup_20260627_235628.json
case_base.json diperbarui -> D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\processed\case_base.json

Sebelum retain         : 150 kasus
Di-update (enrichment) : 28 kasus  ← amar_putusan terverifikasi
Ditambah (entry baru)  : 0 kasus  ← kasus dari luar case base
Di-skip (no teks)      : 0
Total sekarang         : 150 kasus

Contoh entry yang di-update:
  case_002     | DITOLAK        | amar: [DITOLAK] Vonis: pelatihan kerja selama 6 (enam) bulan | Ama
  case_004     | DITOLAK        | amar: [DITOLAK] Vonis: pidana penjara selama 4 (emupat) tahun | Am
  case_024     | DITOLAK        | amar: [DITOLAK] Vonis: pidana penjara selama 2 (dua) tahun | Amar:


## 5 · Simpan Retain Log

In [12]:
retain_log = {
    'iterasi'              : 2,
    'timestamp'            : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'case_base_sebelum'    : len(case_base),
    'kandidat_retain'      : len(df_retain_kandidat),
    'entry_diupdate'       : n_updated,
    'entry_ditambahkan'    : len(entri_baru),
    'entry_diskip'         : len(entri_skip),
    'case_base_sesudah'    : len(case_base_baru),
    'kriteria': {
        'prediksi_benar'       : True,
        'confidence_threshold' : CONFIDENCE_THRESHOLD,
        'kolom_prediksi'       : KOLOM_PRED,
        'update_existing'      : True,
    },
    'detail_updated': [
        {
            'case_id'    : e['case_id'],
            'outcome'    : e['outcome_kasasi'],
            'svm_conf'   : e.get('_retain_svm_conf', 0),
            'aksi'       : 'UPDATE',
        }
        for e in case_base_baru
        if e.get('_retain_iterasi') == 2 and e['case_id'] in id_update
    ],
    'detail_added': [
        {
            'case_id'    : e['case_id'],
            'outcome'    : e['outcome_kasasi'],
            'svm_conf'   : e.get('_retain_svm_conf', 0),
            'aksi'       : 'TAMBAH',
        }
        for e in entri_baru
    ],
}

log_path = METRICS_DIR / 'retain_log.json'
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(retain_log, f, ensure_ascii=False, indent=2)
print(f'retain_log.json disimpan -> {log_path}')
print()

# Simpan CSV
detail_all = retain_log['detail_updated'] + retain_log['detail_added']
if detail_all:
    df_log_retain = pd.DataFrame(detail_all)
    log_csv_path  = METRICS_DIR / 'retain_log.csv'
    df_log_retain.to_csv(log_csv_path, index=False, encoding='utf-8-sig')
    print(f'retain_log.csv disimpan -> {log_csv_path}')
    print()
    print(df_log_retain.to_string(index=False))
else:
    print('Tidak ada kasus yang di-retain.')

retain_log.json disimpan -> D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\eval\metrics\retain_log.json

retain_log.csv disimpan -> D:\Documents\SEMESTER6\Penalaran Komputer\Tugas 3(31)\data\eval\metrics\retain_log.csv

 case_id    outcome  svm_conf   aksi
case_002    DITOLAK    0.8868 UPDATE
case_004    DITOLAK    0.8855 UPDATE
case_024    DITOLAK    0.8845 UPDATE
case_026    DITOLAK    0.8893 UPDATE
case_029    DITOLAK    0.8859 UPDATE
case_035    DITOLAK    0.8817 UPDATE
case_038    DITOLAK    0.8861 UPDATE
case_044    DITOLAK    0.8887 UPDATE
case_048    DITOLAK    0.8908 UPDATE
case_056    DITOLAK    0.8874 UPDATE
case_060    DITOLAK    0.8917 UPDATE
case_067    DITOLAK    0.8908 UPDATE
case_068    DITOLAK    0.8902 UPDATE
case_069    DITOLAK    0.8802 UPDATE
case_073    DITOLAK    0.8919 UPDATE
case_078 DIKABULKAN    0.9035 UPDATE
case_091    DITOLAK    0.8828 UPDATE
case_092 DIPERBAIKI    0.8845 UPDATE
case_105    DITOLAK    0.8942 UPDATE
case_106    DITOLAK    0.890

## 7 · Alur B — Revise & Retain 5 Kasus Baru Konstruktif

Alur ini mensimulasikan siklus CBR untuk kasus yang **betul-betul baru** (tidak ada
dalam case base). Ground truth ditetapkan berdasarkan logika hukum.

Kriteria retain sama dengan Alur A:
- Prediksi Hybrid == Ground Truth ✓
- SVM confidence >= 0.40 ✓  
- case_id belum ada di case_base ✓ (kasus baru — selalu terpenuhi)

Kasus yang berhasil diretain akan memperluas case base dengan preseden baru
yang sebelumnya tidak terwakili — terutama berguna untuk memperkuat kelas DIKABULKAN
dan DIPERBAIKI yang jumlahnya sangat sedikit.


In [13]:
# ── Alur B: Retain 5 Kasus Baru dari NB04 ───────────────────────────────────
# KASUS_BARU dan demo_results_baru harus sudah ada dari NB04 (run NB04 dulu)
# Atau definisikan ulang di sini jika NB05 dijalankan mandiri

# Cek apakah demo_results_baru tersedia dari NB04
try:
    _ = demo_results_baru
    print("✓ demo_results_baru tersedia dari NB04")
except NameError:
    print("⚠ demo_results_baru tidak tersedia — definisikan KASUS_BARU dan jalankan NB04 dulu")
    print("  Atau jalankan NB04 terlebih dahulu dalam session yang sama")
    demo_results_baru = []   # fallback kosong

print()
print("=" * 60)
print("ALUR B — RETAIN KASUS BARU KONSTRUKTIF")
print("=" * 60)

retained_baru    = []
not_retained_baru = []

for r in demo_results_baru:
    case_id  = r['case_id']
    gt       = r['ground_truth']
    pred     = r['prediksi_hybrid']
    conf     = r['svm_confidence']
    benar    = r['benar']

    # Kriteria retain
    crit_benar = benar                        # prediksi tepat
    crit_conf  = conf >= CONFIDENCE_THR       # confidence cukup
    crit_baru  = case_id not in case_base_map # belum ada di case base

    layak = crit_benar and crit_conf and crit_baru

    status = "RETAIN ✓" if layak else "SKIP   ✗"
    alasan = []
    if not crit_benar: alasan.append(f"prediksi salah ({pred} != {gt})")
    if not crit_conf:  alasan.append(f"conf rendah ({conf:.3f} < {CONFIDENCE_THR})")
    if not crit_baru:  alasan.append("sudah ada di case base")

    print(f"  {status} | {case_id} | GT={gt} | Pred={pred} | conf={conf:.3f}")
    if alasan:
        print(f"           Alasan skip: {', '.join(alasan)}")

    if layak:
        # Susun entry baru untuk case base
        # Ambil teks query dari KASUS_BARU
        teks_q = next((k['teks_query'] for k in KASUS_BARU if k['case_id']==case_id), '')
        keterangan = next((k['keterangan'] for k in KASUS_BARU if k['case_id']==case_id), '')

        entry_baru = {
            'case_id'         : case_id,
            'nomor_perkara'   : f'[Konstruktif] {case_id}',
            'outcome_kasasi'  : gt,
            'teks_query'      : teks_q,
            'catatan'         : f'Kasus konstruktif — {keterangan}',
            'iterasi'         : 'open_world_demo',
            'svm_confidence'  : conf,
        }

        retained_baru.append(entry_baru)
        # Optional: tambahkan ke case_base_map untuk sesi ini (tidak permanen)
        # case_base_map[case_id] = entry_baru
    else:
        not_retained_baru.append(case_id)

print()
print(f"Kasus diretain    : {len(retained_baru)} dari {len(demo_results_baru)}")
print(f"Kasus di-skip     : {len(not_retained_baru)}")
print()

if retained_baru:
    print("Detail kasus yang diretain:")
    for e in retained_baru:
        print(f"  {e['case_id']}: {e['outcome_kasasi']} (conf={e['svm_confidence']:.3f})")
        print(f"    → {e['catatan']}")
    print()
    print("Catatan: Entry baru ini dapat ditambahkan ke case_base.json")
    print("         untuk memperkuat preseden kelas minoritas (DIKABULKAN, DIPERBAIKI)")
    print()
    print("Distribusi outcome kasus baru yang diretain:")
    from collections import Counter
    dist = Counter(e['outcome_kasasi'] for e in retained_baru)
    for outcome, cnt in dist.items():
        print(f"  {outcome}: {cnt} kasus")

else:
    print("Tidak ada kasus baru yang memenuhi kriteria retain.")
    print("Ini normal jika model tidak yakin (conf < 0.40) atau prediksi salah.")
    print("Untuk kelas DIKABULKAN dan DIPERBAIKI, scraping kasus tambahan")
    print("dari MA RI sangat direkomendasikan untuk memperkuat case base.")

print()
print("=" * 60)
print("RINGKASAN KEDUA ALUR RETAIN")
print("=" * 60)
print(f"  Alur A (test set)   : lihat Cell 4-6 di atas")
print(f"  Alur B (kasus baru) : {len(retained_baru)} retained, {len(not_retained_baru)} di-skip")
print()
print("Implikasi untuk pengembangan:")
print("  - Kasus DIKABULKAN dan DIPERBAIKI yang berhasil diretain memperkaya")
print("    case base minoritas yang saat ini sangat terbatas (5 dan 10 kasus)")
print("  - Setiap penambahan 1 kasus DIKABULKAN meningkatkan coverage retrieval")
print("    untuk kelas yang paling sulit diprediksi secara signifikan")


✓ demo_results_baru tersedia dari NB04

ALUR B — RETAIN KASUS BARU KONSTRUKTIF

Kasus diretain    : 0 dari 0
Kasus di-skip     : 0

Tidak ada kasus baru yang memenuhi kriteria retain.
Ini normal jika model tidak yakin (conf < 0.40) atau prediksi salah.
Untuk kelas DIKABULKAN dan DIPERBAIKI, scraping kasus tambahan
dari MA RI sangat direkomendasikan untuk memperkuat case base.

RINGKASAN KEDUA ALUR RETAIN
  Alur A (test set)   : lihat Cell 4-6 di atas
  Alur B (kasus baru) : 0 retained, 0 di-skip

Implikasi untuk pengembangan:
  - Kasus DIKABULKAN dan DIPERBAIKI yang berhasil diretain memperkaya
    case base minoritas yang saat ini sangat terbatas (5 dan 10 kasus)
  - Setiap penambahan 1 kasus DIKABULKAN meningkatkan coverage retrieval
    untuk kelas yang paling sulit diprediksi secara signifikan


## 6 · Ringkasan Iterasi & Rekomendasi

In [14]:
('RINGKASAN REVISI & RETAIN — ITERASI 2')
print(f'Case base sebelum  : {len(case_base):>4} kasus')
print(f'Di-update          : {n_updated:>4} kasus  (enrichment amar terverifikasi)')
print(f'Ditambah baru      : {len(entri_baru):>4} kasus')
print(f'Case base sesudah  : {len(case_base_baru):>4} kasus')
print()

df_cb_baru = pd.DataFrame(case_base_baru)
print('Distribusi outcome case base setelah retain:')
dist = df_cb_baru['outcome_kasasi'].value_counts()
total = len(df_cb_baru)
for outcome, n in dist.items():
    bar = '#' * int(n / total * 30)
    print(f'  {outcome:<14}: {n:>3}  {bar}')
print()

if n_updated > 0:
    sample_updated = [e for e in case_base_baru if e.get('_retain_iterasi') == 2 and e['case_id'] in id_update][:5]
    print(f'Contoh kasus yang di-update ({min(5,n_updated)} dari {n_updated}):')
    for e in sample_updated:
        print(f"  {e['case_id']:<12} | {e['outcome_kasasi']:<14} | conf={e.get('_retain_svm_conf',0):.4f}")
    print()

print('Output:')
print(f'  data/processed/case_base.json          <- case base diperbarui ({len(case_base_baru)} kasus)')
print(f'  data/case_base_backup_*.json <- backup case base lama')
print(f'  data/eval/metrics/retain_log.json    <- log retain (JSON)')
print(f'  data/eval/metrics/retain_log.csv     <- log retain (CSV)')
print()
print('Rekomendasi iterasi berikutnya:')
print('  1. Jalankan ulang NB03 — case_base.json sudah diperbarui.')
print('     Entry yang di-update kini punya amar_putusan lebih lengkap.')
print('  2. Amar terverifikasi memperkaya representasi teks sehingga')
print('     retrieval kelas DIKABULKAN dan DIPERBAIKI bisa lebih akurat.')
print('  3. Ulangi siklus: NB03 → NB04 → NB05 → NB06.')

Case base sebelum  :  150 kasus
Di-update          :   28 kasus  (enrichment amar terverifikasi)
Ditambah baru      :    0 kasus
Case base sesudah  :  150 kasus

Distribusi outcome case base setelah retain:
  DITOLAK       : 124  ########################
  TIDAK TERDETEKSI:  11  ##
  DIPERBAIKI    :  10  ##
  DIKABULKAN    :   5  #

Contoh kasus yang di-update (5 dari 28):
  case_002     | DITOLAK        | conf=0.8868
  case_004     | DITOLAK        | conf=0.8855
  case_024     | DITOLAK        | conf=0.8845
  case_026     | DITOLAK        | conf=0.8893
  case_029     | DITOLAK        | conf=0.8859

Output:
  data/processed/case_base.json          <- case base diperbarui (150 kasus)
  data/case_base_backup_*.json <- backup case base lama
  data/eval/metrics/retain_log.json    <- log retain (JSON)
  data/eval/metrics/retain_log.csv     <- log retain (CSV)

Rekomendasi iterasi berikutnya:
  1. Jalankan ulang NB03 — case_base.json sudah diperbarui.
     Entry yang di-update kini punya ama